## make clips

In [9]:
import json

# Load the result dictionary from the JSON file
with open('/Users/kesiyun/Desktop/00workspace/00sequenceresult.json', 'r') as f:
    loaded_result = json.load(f)

#for key, value in loaded_result.items():
    #print(key,': ',value)



In [30]:
## make clips

import re

def parse_timestamps(timesec):
    pattern = r'([A-Z])\((\d+):(\d+)\)'
    matches = re.findall(pattern, timesec)
    timestamps = []
    for match in matches:
        label, minutes, seconds = match
        total_seconds = int(minutes) * 60 + int(seconds)
        timestamps.append((label, total_seconds))
    return timestamps

def create_virtual_clips(timesec, length_of_clips):
    timestamps = parse_timestamps(timesec)
    intervals = []

    # Build intervals: (label, start_time, end_time)
    for i in range(len(timestamps)):
        label, start = timestamps[i]
        end = timestamps[i + 1][1] if i + 1 < len(timestamps) else timestamps[-1][1] + length_of_clips
        intervals.append((label, start, end))

    # Ensure initial label 'R' is applied from time 0 until the first timestamp
    intervals.insert(0, ('R', 0, timestamps[0][1]))

    # Determine total duration
    total_duration = intervals[-1][2]
    clips = []

    # For each clip, collect active labels
    for clip_start in range(0, total_duration, length_of_clips):
        clip_end = clip_start + length_of_clips
        active_labels = set()
        for label, start, end in intervals:
            if start < clip_end and end > clip_start:
                active_labels.add(label)
        clips.append(''.join(sorted(active_labels)))

    return clips


def create_virtual_clips_start(timesec, length_of_clips):
    timestamps = parse_timestamps(timesec)
    clips = []
    current_time = 0
    last_label = 'R'  # Default label at the start

    for label, timestamp in timestamps:
        while current_time + length_of_clips <= timestamp:
            clips.append(last_label)
            current_time += length_of_clips
        last_label = label

    # Fill remaining clips until the last timestamp
    while current_time <= timestamps[-1][1]:
        clips.append(last_label)
        current_time += length_of_clips

    return clips


# Example usage
timesec = "R P(0:40) A(0:44)B(0:51)C(1:06)C(1:50) P(1:54) A(2:21)C(2:28)A(3:25)C(3:30)"
length_of_clips = 4
clips = create_virtual_clips(timesec, length_of_clips)
print("Record all labels covered:", clips)

clips_single = create_virtual_clips_start(timesec, length_of_clips)
print("Starting label only:", clips_single)

Record all labels covered: ['R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'P', 'A', 'AB', 'B', 'B', 'B', 'BC', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'CP', 'P', 'P', 'P', 'P', 'P', 'P', 'AP', 'A', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'AC', 'AC', 'C']
Starting label only: ['R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'P', 'A', 'B', 'B', 'B', 'B', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'P', 'P', 'P', 'P', 'P', 'P', 'P', 'A', 'A', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'A', 'C']


In [49]:
import json

def create_virtual_clip_refs(base_dir,length_of_clips):
    virtual_clip_dict={}
    for key, value in loaded_result.items():
        #print(key,': ',value)
        virtual_clip_dict[key]={'num_of_clips':0,'list_of_clips':[]}
        virtual_clip_dict[key]['list_of_clips']=create_virtual_clips(loaded_result[key]['timesec'], length_of_clips) 
        virtual_clip_dict[key]['num_of_clips']=len(virtual_clip_dict[key]['list_of_clips'])
    
    #save to file
    with open(base_dir+'/00virtual_clips_'+str(length_of_clips)+'s.json', 'w') as f:
        json.dump(virtual_clip_dict, f)


    

In [50]:
length_of_clips=5
base_dir='/Users/kesiyun/Desktop/00workspace'
create_virtual_clip_refs(base_dir,length_of_clips)

## split dataset

In [1]:

import random

def split_videos(virtual_clip_dict):
    # Calculate total number of clips
    total_clips = sum(entry['num_of_clips'] for entry in virtual_clip_dict.values())
    
    # Calculate target number of clips for testing set (20% of total)
    target_clips = total_clips * 0.20
    
    # Initialize lists for testing and training sets
    testing_set = []
    training_set = list(virtual_clip_dict.keys())
    
    # Initialize accumulated clips counter
    accumulated_clips = 0
    
    # Randomly shuffle the keys to ensure random selection
    keys = list(virtual_clip_dict.keys())
    random.shuffle(keys)
    
    # Iterate through the shuffled keys and accumulate clips for testing set
    for key in keys:
        if accumulated_clips < target_clips:
            testing_set.append(key)
            accumulated_clips += virtual_clip_dict[key]['num_of_clips']
            training_set.remove(key)
        else:
            break
    
    return testing_set, training_set

# Example usage
virtual_clip_dict_sample = {
    'video1': {'num_of_clips': 52, 'list_of_clips': ['clip1', 'clip2', 'clip3']},
    'video2': {'num_of_clips': 100, 'list_of_clips': ['clip4', 'clip5', 'clip6']},
    'video3': {'num_of_clips': 200, 'list_of_clips': ['clip7', 'clip8', 'clip9']},
    # Add more entries as needed
}

testing_set1, training_set1 = split_videos(virtual_clip_dict_sample)

print("Testing Set1:", testing_set1)
print("Training Set1:", training_set1)


Testing Set1: ['video2']
Training Set1: ['video1', 'video3']


In [52]:
# Load the virtual clip dictionary from the JSON file
with open('/Users/kesiyun/Desktop/00workspace/00virtual_clips_5s.json', 'r') as f:
    loaded_virtual_clip_dict = json.load(f)

total_clips=0
key_count=0
for key, value in loaded_virtual_clip_dict.items():
    #print(key,": ",value['num_of_clips'] )
    total_clips+=value['num_of_clips']
    key_count+=1
print(total_clips, 'clips in',key_count,'videos (average',total_clips/key_count,'each)')


6763 clips in 57 videos (average 118.64912280701755 each)


In [63]:
#testing_set, training_set = split_videos(loaded_virtual_clip_dict)#safety catch
print("Testing Set:", testing_set)
print("Training Set:", training_set)

result = {
    'testing_set': testing_set,
    'training_set': training_set
}

with open('xxx00video_split_5s.json', 'w') as f:#safety catch
    json.dump(result, f, indent=4)


Testing Set: ['13A', '17B', '6B', '3B', '17H', '17G', '8B', '14C', '14D', '10C', '1F', '3D', '15C', '17A', '10A']
Training Set: ['1A', '1B', '1C', '1D', '1E', '2A', '3A', '3C', '4A', '4B', '4C', '4D', '4E', '4F', '5A', '6A', '6C', '6D', '7A', '8A', '9A', '10B', '11A', '11B', '11C', '11D', '12A', '12B', '14A', '14B', '14E', '15A', '15B', '15D', '16A', '17C', '17D', '17E', '17F', '17I', '18A', '18B']


In [68]:
## load dataset profiles
import json

with open('00video_split_5s.json', 'r') as f:
    data = json.load(f)

testing_set = data['testing_set']
training_set = data['training_set']

print("Testing Set:", testing_set)
total_clips=0
key_count=0
for videos in testing_set:
    total_clips+=loaded_virtual_clip_dict[videos]['num_of_clips']
    key_count+=1
print(total_clips, 'clips in',key_count,'videos (average',total_clips/key_count,'each) \n')

print("Training Set:", training_set)
total_clips=0
key_count=0
for videos in training_set:
    total_clips+=loaded_virtual_clip_dict[videos]['num_of_clips']
    key_count+=1
print(total_clips, 'clips in',key_count,'videos (average',total_clips/key_count,'each) \n')

Testing Set: ['13A', '17B', '6B', '3B', '17H', '17G', '8B', '14C', '14D', '10C', '1F', '3D', '15C', '17A', '10A']
1370 clips in 15 videos (average 91.33333333333333 each) 

Training Set: ['1A', '1B', '1C', '1D', '1E', '2A', '3A', '3C', '4A', '4B', '4C', '4D', '4E', '4F', '5A', '6A', '6C', '6D', '7A', '8A', '9A', '10B', '11A', '11B', '11C', '11D', '12A', '12B', '14A', '14B', '14E', '15A', '15B', '15D', '16A', '17C', '17D', '17E', '17F', '17I', '18A', '18B']
5393 clips in 42 videos (average 128.4047619047619 each) 



In [95]:
def timestamp_to_clip_index(timestamp, length_of_clips):
    # Parse the timestamp string (e.g., "0:03")
    minutes, seconds = map(int, timestamp.split(':'))
    total_seconds = minutes * 60 + seconds

    # Calculate which clip this timestamp falls into
    clip_index = total_seconds // length_of_clips

    return clip_index

# Example usage:
#print(timestamp_to_clip_index("0:03", 4))  # ➜ 0
#print(timestamp_to_clip_index("0:05", 3))  # ➜ 1
#print(timestamp_to_clip_index("0:10", 5))  # ➜ 2


def get_timestamp(length_of_clip, clip_index):
    total_seconds = length_of_clip * clip_index
    minutes = total_seconds // 60
    seconds = total_seconds % 60
    return f"{minutes}:{seconds:02d}"


# Examples:
#print(get_timestamp(5, 0))  # Output: 0:00
#print(get_timestamp(5, 1))  # Output: 0:05
#print(get_timestamp(5, 3))  # Output: 0:15


In [101]:
#max diff feature

import cv2
import numpy as np

def find_contractions(list_of_clips):
    C_locations=[]
    for i in range(len(list_of_clips)):
        if 'C' in list_of_clips[i]:
            C_locations.append(i)
    return C_locations



def timestamp_to_seconds(timestamp):
    minutes, seconds = map(int, timestamp.split(':'))
    return minutes * 60 + seconds

def max_difference_feature(video_path, start_timestamp, clip_duration):
    # Convert start timestamp to seconds
    start_seconds = timestamp_to_seconds(start_timestamp)
    
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    
    # Get the frame rate of the video
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    # Calculate the start frame and end frame
    start_frame = int(start_seconds * fps)
    end_frame = int((start_seconds + clip_duration) * fps)
    
    # Set the video to start at the start frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    # Initialize variables
    prev_frame = None
    max_diff = 0
    
    # Loop through the frames in the clip
    for i in range(start_frame, end_frame):
        ret, frame = cap.read()
        if not ret:
            break
        
        # Convert the frame to grayscale
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        if prev_frame is not None:
            # Calculate the absolute difference between consecutive frames
            diff = cv2.absdiff(prev_frame, gray_frame)
            # Find the maximum pixel difference in the current difference frame
            max_diff = max(max_diff, np.max(diff))
        
        # Update the previous frame
        prev_frame = gray_frame
    
    # Release the video capture object
    cap.release()
    
    return max_diff


In [102]:
#load video and profiles
import json
with open('/Users/kesiyun/Desktop/00workspace/00sequenceresult.json', 'r') as f:
    loaded_sequence = json.load(f)
with open('/Users/kesiyun/Desktop/00workspace/00virtual_clips_5s.json', 'r') as f:
    loaded_virtual_clip_dict = json.load(f)
with open('00video_split_5s.json', 'r') as f:
    data = json.load(f)
testing_set = data['testing_set']
training_set = data['training_set']

length_of_clip=5
videos_dir='/Users/kesiyun/Desktop/00workspace/videos' ####
video_name=training_set[0]#'1A' ####

avi_path=videos_dir+'/'+video_name+'.avi'
print('Video file:',avi_path)
print('Sequence:', loaded_sequence[video_name])
print('Clips (',length_of_clip,'s):', loaded_virtual_clip_dict[video_name],)
list_of_contractions=find_contractions(loaded_virtual_clip_dict[video_name]['list_of_clips'])
print('Contractions in:',list_of_contractions)




Video file: /Users/kesiyun/Desktop/00workspace/videos/1A.avi
Sequence: {'abstract': 'R P ACD', 'timesec': 'R P(0:39) A(1:38)C(1:52)D(4:25)'}
Clips ( 5 s): {'num_of_clips': 54, 'list_of_clips': ['R', 'R', 'R', 'R', 'R', 'R', 'R', 'PR', 'P', 'P', 'P', 'P', 'P', 'P', 'P', 'P', 'P', 'P', 'P', 'AP', 'A', 'A', 'AC', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'D']}
Contractions in: [22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52]


In [104]:
features_contraction=[]
features_non_contraction=[]
for i in range (len(loaded_virtual_clip_dict[video_name]['list_of_clips'])):
    if i in list_of_contractions:
        start_timestamp = get_timestamp(length_of_clip, i) #'1:50'
        feature = max_difference_feature(avi_path, start_timestamp, length_of_clip)
        #print(f"The max difference feature for the clip is: {feature}")
        features_contraction.append(feature)
    else:
        start_timestamp = get_timestamp(length_of_clip, i) #'1:50'
        feature = max_difference_feature(avi_path, start_timestamp, length_of_clip)
        #print(f"The max difference feature for the clip is: {feature}")
        features_non_contraction.append(feature)

print(features_contraction)
print(features_non_contraction)

#max difference feature might be good for catching the frames with faster movements... but the current dataset label "contraction" seemed to describe a "contracted" shape/behaviour of stentors rather than the movement when they were contracting. Thus there seemed to be no great difference for the max difference features captured over the C labels and non-C labels.


[207, 202, 208, 201, 212, 196, 202, 194, 150, 140, 140, 138, 142, 181, 111, 117, 172, 152, 180, 181, 195, 183, 188, 193, 185, 164, 177, 185, 183, 191, 200]
[154, 118, 110, 124, 103, 113, 104, 152, 217, 229, 165, 180, 181, 170, 216, 179, 185, 177, 179, 174, 166, 217, 146]
